In [ ]:
// ==== Setup ====

import java.util.Random;

int comparisons, swaps;   // reset before each run

double[] example = {13, 6, 2, 7, 11, 4, 8, 15, 9, 3};   // same running example as last time

# Quicksort

Another *divide and conquer* algorithm.

No merge step, and no second array: the partition rearranges the array in place.

1. **Base case**: What instances of the problem are trivial to solve?
2. **Recursive call**: Given a generic instance, how can I make it simpler, or "closer" to one of the trivial instances?
3. **Combine the results**: How can I use the answer of the simpler instances to solve the harder one?
4. **Check for termination**: Does every path in my algorithm eventually reach a base case?

## Partitioning

Pivot moves to the rightmost spot first, out of the way. Two pointers close in from both ends: `left` stops on something too big, `right` stops on something too small, swap. When they meet, drop the pivot there - it is now in its final position.

In [ ]:
void swap(double[] array, int i, int j) {
    double t = array[i]; array[i] = array[j]; array[j] = t;
}

int partition(double[] array, int left, int right, int pivotIndex) {
    int pivotPos = right;
    swap(array, pivotIndex, pivotPos);      // move the pivot to the rightmost position
    double pivot = array[pivotPos];
    right--;

    while (right >= left) {
        while (array[left] < pivot) { comparisons++; left++; }
        while (right >= left && array[right] >= pivot) { comparisons++; right--; }
        if (right > left) { swap(array, left, right); swaps++; }
    }

    swap(array, left, pivotPos);            // move the pivot to its final location
    swaps++;
    return left;
}

## quickSort

Same wrapper shape as `reverse` and `binarySearch2` last lecture: a public entry point guards the trivial case, a private recursive method assumes it is safe.

Here the guard moves one level further up again - the **caller** decides whether a side is even worth recursing into, rather than checking it on entry.

In [ ]:
int findPivotFirst(double[] array, int left, int right) {
    return left;                            // naive: always the first element
}

void quickSortRec(double[] array, int left, int right) {
    int pivotIndex = findPivotFirst(array, left, right);
    int newPivotIndex = partition(array, left, right, pivotIndex);
    if (newPivotIndex - left > 1)  quickSortRec(array, left, newPivotIndex - 1);
    if (right - newPivotIndex > 1) quickSortRec(array, newPivotIndex + 1, right);
}

void quickSort(double[] array) {
    if (array.length < 2) return;
    quickSortRec(array, 0, array.length - 1);
}

double[] qck = example.clone();
comparisons = 0; swaps = 0;
quickSort(qck);
System.out.println(java.util.Arrays.toString(qck));
System.out.println("comparisons=" + comparisons + "  swaps=" + swaps);

## Picking the pivot

`partition` doesn't care which element you feed it as the pivot - but the split it produces does. What would be the best pivot? Why was merge sort `O(NlogN)`?


### Common strategies

- **First element** - free, no guarantee about the split. Equivalently we can pick the last.
- **Middle element** - also free, dodges one obvious failure mode.
- **Random element** - no single input can be prepared in advance to defeat it.
- **Median-of-three** - sample first, middle, last; take their median. A couple of extra comparisons, more reliably balanced than any single fixed position.
- Median-of-five etc...

### A bad pivot on sorted input

Always the first element. If the array is already sorted, every partition splits N elements into **0 and N-1**... the worst possible split every time.

In [ ]:
System.out.printf("%8s %12s %10s%n", "N", "comparisons", "ratio");
double prevSorted = 0;
for (int size = 250; size <= 4000; size *= 2) {
    double[] array = new double[size];
    for (int i = 0; i < size; i++) array[i] = i;      // already sorted
    comparisons = 0;
    quickSort(array);
    System.out.printf("%8d %12.0f %10s%n", size, (double) comparisons,
                       prevSorted == 0 ? "-" : String.format("%.2f", comparisons / prevSorted));
    prevSorted = comparisons;
}

In [ ]:
Random pivotRng = new Random(7);
int findPivotRandom(double[] array, int left, int right) {
    return left + pivotRng.nextInt(right - left + 1);
}

void quickSortRecRandom(double[] array, int left, int right) {
    int pivotIndex = findPivotRandom(array, left, right);
    int newPivotIndex = partition(array, left, right, pivotIndex);
    if (newPivotIndex - left > 1)  quickSortRecRandom(array, left, newPivotIndex - 1);
    if (right - newPivotIndex > 1) quickSortRecRandom(array, newPivotIndex + 1, right);
}

System.out.printf("%8s %12s %10s%n", "N", "comparisons", "ratio");
double prevRand = 0;
for (int size = 250; size <= 4000; size *= 2) {
    double[] array = new double[size];
    for (int i = 0; i < size; i++) array[i] = i;      // same sorted input
    comparisons = 0;
    quickSortRecRandom(array, 0, array.length - 1);
    System.out.printf("%8d %12.0f %10s%n", size, (double) comparisons,
                       prevRand == 0 ? "-" : String.format("%.2f", comparisons / prevRand));
    prevRand = comparisons;
}

Same sorted input, only the pivot strategy changed. Worst case is a property of the pivot choice, not just the algorithm.

## A closer look at quicksort

```
1  quickSort ( array )
2    quickSortRec( array, 0, array.length - 1 )
```

```
3  quickSortRec (array, left, right)
4    pivotIndex = findpivot(array, left, right)   // use any method
5    newPivotIndex = partition( array, left, right, pivotIndex )
6    if ( newPivotIndex - left > 1 )
7      quickSortRec( array, left, newPivotIndex - 1 )
8    if ( right - newPivotIndex > 1 )
9      quickSortRec( array, newPivotIndex + 1, right )
```

For these questions, `findpivot` returns `(left+right)/2` - the middle index. `partition` returns the pivot's final position.

- What does `array` look like after the very first call to `partition` on line 5 completes?
- After the second call to `partition`?
- After the recursive call on line 7 completely finishes?
- Would the final result change if lines 6 and 8 (the `if`s, bodies included) were swapped?
- What breaks if the `if` checks on lines 6 and 8 are removed, keeping only the recursive calls?

## Hybrid sorts

In practical implementations it is common to mix-and-match sorting algorithms according to some threshold. Examples include switching to quadratic sorts for small input sizes, or switching to a linearithmic sort if quicksort exceeds a certain recursion depth to guarantee O(NlogN) time ("introsort").

## Properties of sorting algorithms

- **Adaptive** - faster on data that is already partly sorted?
- **Stable** - do equal keys keep their original relative order?
- **In-place** - O(1) extra memory, no second array?

| | adaptive | stable | in-place |
|---|---|---|---|
| bubble | no | yes | yes |
| selection | no | no | yes |
| insertion | **yes** | yes | yes |
| merge | no | yes | **no** |
| quicksort | no | no | yes |

When do these properties matter?
What is the complexity of insertion sort on sorted input?

## Comparable and Comparator

Everything above sorts `double[]`. Real code might sort `Student[]`, `Point[]`... but a `double` has a built-in `<` while an object does not.

### `Comparable<T>`

The type declares its own natural order, once, inside the class.

In [ ]:
class Student implements Comparable<Student> {
    String name;
    double gpa;
    Student(String name, double gpa) { this.name = name; this.gpa = gpa; }

    // returns a negative integer if this < other
    // returns zero if this == other
    // returns a positive integer if this > other
    @Override
    public int compareTo(Student other) { return Double.compare(gpa, other.gpa); }

    @Override
    public String toString() { return name + "(" + gpa + ")"; }
}

Student[] roster = {
    new Student("Ann", 3.2), new Student("Bo", 3.9), new Student("Cy", 2.8)
};

### `Comparator<T>`

An order supplied from outside. The class itself is not "aware" of it.

In [ ]:
import java.util.Comparator;   // Comparable needed no import - java.lang is automatic. Comparator lives in java.util.

class ByName implements Comparator<Student> {
    // same as compareTo: negative/zero/positive for a <, ==, > b
    @Override
    public int compare(Student a, Student b) { return a.name.compareTo(b.name); }
}

Comparator<Student> byName = new ByName();

### Sorting and searching, with `Comparable`

A small class bound to `T`: `array[k].compareTo(currentElement)` - the type compares itself. Same two algorithms as before, generalized.

In [ ]:
class ComparableSorter<T extends Comparable<T>> {
    void insertionSort(T[] array) {
        for (int i = 1; i < array.length; i++) {
            T currentElement = array[i];
            int k;
            for (k = i - 1; k >= 0; k--) {
                if (array[k].compareTo(currentElement) <= 0) // <===
                    break;
                array[k + 1] = array[k];
            }
            array[k + 1] = currentElement;
        }
    }

    int binarySearch(T[] array, T key) {
        int left = 0, right = array.length - 1;
        while (left <= right) {
            int mid = (left + right) / 2;
            int c = array[mid].compareTo(key); // <===
            if      (c < 0) left = mid + 1;
            else if (c > 0) right = mid - 1;
            else            return mid;
        }
        return -1;
    }
}

Student[] byGpa = roster.clone();
ComparableSorter<Student> naturalOrder = new ComparableSorter<>();
naturalOrder.insertionSort(byGpa);
System.out.println("natural order (gpa) : " + java.util.Arrays.toString(byGpa));
System.out.println(naturalOrder.binarySearch(byGpa, new Student("Bo", 3.9)));

### Sorting and searching, with `Comparator`

Independent implementation, same shape: `cmp.compare(array[k], currentElement)` - an object supplied from outside does the comparing instead.

In [ ]:
class ComparatorSorter<T> {
    Comparator<T> cmp;
    ComparatorSorter(Comparator<T> cmp) { this.cmp = cmp; }

    void insertionSort(T[] array) {
        for (int i = 1; i < array.length; i++) {
            T currentElement = array[i];
            int k;
            for (k = i - 1; k >= 0; k--) {
                if (cmp.compare(array[k], currentElement) <= 0) // <===
                    break;
                array[k + 1] = array[k];
            }
            array[k + 1] = currentElement;
        }
    }

    int binarySearch(T[] array, T key) {
        int left = 0, right = array.length - 1;
        while (left <= right) {
            int mid = (left + right) / 2;
            int c = cmp.compare(array[mid], key); // <===
            if      (c < 0) left = mid + 1;
            else if (c > 0) right = mid - 1;
            else            return mid;
        }
        return -1;
    }
}

Student[] byNameArr = roster.clone();
ComparatorSorter<Student> byNameSorter = new ComparatorSorter<>(byName);
byNameSorter.insertionSort(byNameArr);
System.out.println("by name             : " + java.util.Arrays.toString(byNameArr));
System.out.println(byNameSorter.binarySearch(byNameArr, new Student("Cy", 0)));